<a href="https://colab.research.google.com/github/wyclliffe/science-corp-website/blob/main/wycliffe_data_science_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

i will start by installing the necessary library for data extraction

In [67]:

!pip install -q requests beautifulsoup4 pandas openpyxl


i will import the library to help in data extraction

In [68]:

import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

from datetime import date
from urllib.parse import urlparse, urldefrag
from urllib.robotparser import RobotFileParser

print("All libraries imported successfully!")


All libraries imported successfully!


i will define the required 29 column

In [69]:
columns = [
    "Scholarship name",
    "Scholarship provider/organization",
    "Provider type",
    "Official scholarship URL",
    "Discovery source",
    "State/region",
    "National vs. state/local",
    "Eligible grade levels",
    "Middle school / high school / both",
    "Scholarship category",
    "Field/area of interest",
    "Award amount",
    "Number of awards (if known)",
    "Renewable (Yes/No/Unknown)",
    "Maximum potential value",
    "Application open date (if known)",
    "Application deadline",
    "Residency requirement",
    "Minimum GPA (if any)",
    "Financial need required (Yes/No/Unknown)",
    "Essay required",
    "Recommendation required",
    "Transcript/test score required",
    "Community service requirement",
    "Leadership requirement",
    "Other major eligibility requirements",
    "Application method",
    "Verification date",
    "Notes"
]

print("Number of required fields:", len(columns))


Number of required fields: 29


i will create alist of scholarship scholarship websites

In [70]:

# Scholarship website URLs
# Source: Wyclliffe_Magangi_Final_Scholarship_Workbook.xlsx

scholarship_urls = [
    # 1. Aubrey Lee Brooks Scholarship
    "https://www.cfnc.org/pay-for-college/apply-for-financial-aid/aubrey-lee-brooks-scholarship/",

    # 2. Golden LEAF Scholarship
    "https://www.cfnc.org/pay-for-college/apply-for-financial-aid/golden-leaf-colleges-and-universities-scholarship/",

    # 3. M&J Scholarship
    "https://www.northcarolina.edu/students/scholarships/mj-scholarship/",

    # 4. Knights of Pythias Scholarship
    "https://www.nckop.com/scholarship/",

    # 5. DHS Friends Hilda Crocker Wilson Scholarship
    "https://www.nccommunityfoundation.org/scholarship/dhs-friends-hilda-crocker-wilson-scholarship-endowment-fund",

    # 6. C. Reid Ross Rotary Scholarship
    "https://www.nccommunityfoundation.org/scholarships/how-to-apply/other-nccf-scholarships",

    # 7. CASL Scholarship in Memory of Lane Meyer
    "https://www.nccommunityfoundation.org/scholarship/casl-scholarship-fund-in-memory-of-lane-meyer",

    # 8. NSHSS Healthcare Heroes Scholarship
    "https://www.nshss.org/scholarships/",

    # 9. NSHSS STEAM Scholarship
    "https://www.nshss.org/scholarships/s/nshss-steam-scholarship/",

    # 10. Carteret County Retired School Personnel
    "https://www.nccommunityfoundation.org/scholarship/carteret-county-retired-school-personnel-scholarship-endowment",

    # 11. Daughtry Family Foundation Scholarship
    "https://www.cfnc.org/pay-for-college/apply-for-financial-aid/daughtry-family-foundation-scholarship/",

    # 12. Decision Partners Financial Basics Scholarship
    "https://www.cfnc.org/plan-your-future/planning-resources/financial-literacy/financial-basics-scholarship-contest-official-rules/"
]

print("Total scholarship websites:", len(scholarship_urls))


Total scholarship websites: 12


i will create web page extraction

In [71]:

session = requests.Session()

session.headers.update({
    "User-Agent": "ScholarshipResearchStudent/1.0"
})

def extract_webpage(url):

    try:
        parsed = urlparse(url)

        if parsed.scheme != "https":
            print("Skipping non-HTTPS URL:", url)
            return None

        robots = RobotFileParser()
        robots.set_url(
            f"{parsed.scheme}://{parsed.netloc}/robots.txt"
        )
        robots.read()

        if not robots.can_fetch(
            session.headers["User-Agent"], url
        ):
            print("Automated access not permitted:", url)
            return None

        response = session.get(url, timeout=20)
        response.raise_for_status()

        if "text/html" not in response.headers.get(
            "Content-Type", ""
        ).lower():
            print("Page is not HTML:", url)
            return None

        soup = BeautifulSoup(
            response.text, "html.parser"
        )

        for tag in soup([
            "script", "style", "nav", "footer", "header"
        ]):
            tag.decompose()

        heading = soup.find("h1")

        title = (
            heading.get_text(" ", strip=True)
            if heading
            else "Unknown"
        )

        main = (
            soup.find("main")
            or soup.find("article")
            or soup
        )

        text = main.get_text(
            separator=" ", strip=True
        )

        text = re.sub(r"\s+", " ", text)

        return {
            "title": title,
            "text": text,
            "url": response.url
        }


    except requests.exceptions.HTTPError as error:
        print("HTTP error:", url, error)
        return None

    except requests.exceptions.Timeout:
        print("Website timed out:", url)
        return None

    except requests.exceptions.ConnectionError as error:
        print("Connection problem:", url, error)
        return None

    except Exception as error:
        print("Other error:", url, error)
        return None


print("Web extraction function created.")


Web extraction function created.


Extract scholarship award amounts
This function searches the webpage for dollar amounts.

In [72]:

def extract_award_amount(text):

    pattern = r"\$\s?\d{1,3}(?:,\d{3})*(?:\.\d{2})?"

    amounts = re.findall(pattern, text)

    if amounts:
        return ", ".join(dict.fromkeys(amounts[:10]))

    return "Unknown"


sample_text = "Students can receive a $3,500 scholarship."

print(extract_award_amount(sample_text))


$3,500


In [73]:

def extract_award_amount(text):
    patterns = [
        r"(?:scholarship award|award amount|scholarship amount)"
        r"[\s:–-]{0,20}(\$\s?[\d,]+)",

        r"(?:receive|awarded|provides|offers)"
        r".{0,60}?(\$\s?[\d,]+)"
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)

        if match:
            return match.group(1)

    return "Unknown"


Extract scholarship application deadlines

In [74]:

def extract_deadline(text):

    patterns = [
        r"(?:application deadline|deadline|apply by|due date)"
        r"[\s:–-]{0,12}"
        r"([A-Za-z]+\s+\d{1,2},?\s*\d{4})",

        r"(?:application deadline|deadline|apply by|due date)"
        r"[\s:–-]{0,12}"
        r"(\d{1,2}/\d{1,2}/\d{4})"
    ]

    for pattern in patterns:
        match = re.search(
            pattern, text, re.IGNORECASE
        )

        if match:
            return match.group(1)

    return "Unknown"


sample = "Application deadline: March 1, 2027"

print(extract_deadline(sample))


March 1, 2027


Extract GPA requirements

In [75]:

def extract_gpa(text):

    pattern = (
        r"(?:minimum\s+)?GPA"
        r".{0,30}?"
        r"\b[0-4]\.\d{1,2}\b"
    )

    match = re.search(
        pattern, text, re.IGNORECASE
    )

    if match:
        return match.group(0)

    return "Unknown"


sample = "Applicants must have a minimum GPA of 3.0."

print(extract_gpa(sample))


minimum GPA of 3.0


Extract grade levels

In [76]:

def extract_grade_levels(text):

    grades = []

    for grade in range(6, 13):

        pattern = (
            rf"\b(?:grade\s*{grade}|"
            rf"{grade}(?:st|nd|rd|th)\s+grade)\b"
        )

        if re.search(
            pattern, text, re.IGNORECASE
        ):
            grades.append(str(grade))

    if grades:
        return ", ".join(grades)

    return "Unknown"


sample = "Open to students in grades 9, 10, 11 and 12."

print(extract_grade_levels(sample))


Unknown


This basic extractor will find explicitly written grade references such as "grade 9" or "12th grade." It will not reliably identify every grade format or phrases such as "graduating seniors."

Create a scholarship record
This function organizes the extracted information into the 29 required fields.

Extract scholarship information from all websites

In [77]:

def extract_requirement(text, keywords):
    sentences = re.split(r"(?<=[.!?])\s+", text)

    matches = [
        sentence.strip()
        for sentence in sentences
        if any(
            keyword.lower() in sentence.lower()
            for keyword in keywords
        )
    ]

    return " | ".join(matches[:3]) if matches else "Unknown"


def create_scholarship_record(page):

    text = page["text"]

    record = {
        column: "Unknown"
        for column in columns
    }

    record["Scholarship name"] = page["title"]
    record["Official scholarship URL"] = page["url"]
    record["Discovery source"] = "Official scholarship website"

    record["Award amount"] = extract_award_amount(text)
    record["Application deadline"] = extract_deadline(text)
    record["Minimum GPA (if any)"] = extract_gpa(text)
    record["Eligible grade levels"] = extract_grade_levels(text)

    record["Residency requirement"] = extract_requirement(
        text, ["resident", "residency", "county"]
    )
    record["Scholarship provider/organization"] = identify_provider(
    page["url"]
)

    record["Financial need required (Yes/No/Unknown)"] = (
        extract_requirement(
            text, ["financial need", "Pell eligible"]
        )
    )

    record["Essay required"] = extract_requirement(
        text, ["essay", "personal statement"]
    )

    record["Recommendation required"] = extract_requirement(
        text, ["recommendation", "reference letter"]
    )

    record["Transcript/test score required"] = (
        extract_requirement(
            text, ["transcript", "SAT", "ACT score"]
        )
    )

    record["Community service requirement"] = (
        extract_requirement(
            text, ["community service", "volunteer"]
        )
    )

    record["Leadership requirement"] = extract_requirement(
        text, ["leadership", "leader"]
    )

    record["Other major eligibility requirements"] = (
        extract_requirement(
            text, ["eligible", "applicant", "must"]
        )
    )

    record["Application method"] = extract_requirement(
        text, ["apply online", "application portal", "submit"]
    )

    record["Verification date"] = date.today().isoformat()

    record["Notes"] = (
        "Automatically extracted candidate information. "
        "Requirements must be checked against the official page."
    )

    return record


This improves extraction by finding relevant sentences from each scholarship webpage.

In [78]:

scholarship_data = []

for number, url in enumerate(
    scholarship_urls, start=1
):

    print(f"\nProcessing scholarship {number}")

    page = extract_webpage(url)

    if page is not None:

        record = create_scholarship_record(page)

        scholarship_data.append(record)

        print("Name:", record["Scholarship name"])
        print("Award:", record["Award amount"])
        print(
            "Deadline:",
            record["Application deadline"]
        )

    time.sleep(2)

print(
    "\nTotal scholarships extracted:",
    len(scholarship_data)
)



Processing scholarship 1
Automated access not permitted: https://www.cfnc.org/pay-for-college/apply-for-financial-aid/aubrey-lee-brooks-scholarship/

Processing scholarship 2
Automated access not permitted: https://www.cfnc.org/pay-for-college/apply-for-financial-aid/golden-leaf-colleges-and-universities-scholarship/

Processing scholarship 3
Name: M&J Scholarship
Award: $4,000
Deadline: Unknown

Processing scholarship 4
Automated access not permitted: https://www.nckop.com/scholarship/

Processing scholarship 5
Name: DHS Friends Hilda Crocker Wilson Scholarship Endowment Fund
Award: Unknown
Deadline: March 3, 2026

Processing scholarship 6
Name: Other NCCF Scholarships
Award: Unknown
Deadline: Unknown

Processing scholarship 7
Name: CASL Scholarship Fund in Memory of Lane Meyer
Award: Unknown
Deadline: March 15, 2026

Processing scholarship 8
Name: Your Path to Recognition, Scholarship Opportunities, and Success
Award: $1,000
Deadline: Unknown

Processing scholarship 9
Name: NSHSS ST

adding provider information

In [79]:

provider_lookup = {
    "aubrey-lee-brooks-scholarship":
        "Aubrey Lee Brooks Foundation",

    "golden-leaf-colleges-and-universities-scholarship":
        "Golden LEAF Foundation",

    "mj-scholarship":
        "M&J Scholarship donors",

    "nckop.com":
        "Grand Lodge, Knights of Pythias of North Carolina",

    "dhs-friends-hilda-crocker-wilson":
        "Pending sponsor confirmation",

    "other-nccf-scholarships":
        "Pending sponsor confirmation",

    "casl-scholarship-fund":
        "Pending sponsor confirmation",

    "nshss.org":
        "National Society of High School Scholars",

    "carteret-county-retired-school-personnel":
        "Carteret County Retired School Personnel",

    "daughtry-family-foundation":
        "Daughtry Family Foundation",

    "financial-basics-scholarship":
        "Decision Partners"
}


def identify_provider(url):

    url = url.lower()

    for keyword, provider in provider_lookup.items():
        if keyword in url:
            return provider

    return "Unknown"


Convert the data into a Pandas DataFrame
A DataFrame organizes the scholarship information into rows and columns.

In [80]:

df = pd.DataFrame(
    scholarship_data,
    columns=columns
)

print("Dataset dimensions:", df.shape)

display(df.head())


Dataset dimensions: (7, 29)


,Scholarship name,Scholarship provider/organization,Provider type,Official scholarship URL,Discovery source,State/region,National vs. state/local,Eligible grade levels,Middle school / high school / both,Scholarship category,...,Financial need required (Yes/No/Unknown),Essay required,Recommendation required,Transcript/test score required,Community service requirement,Leadership requirement,Other major eligibility requirements,Application method,Verification date,Notes
0,M&J Scholarship,M&J Scholarship donors,Unknown,https://www.northcarolina.edu/students/scholar...,Official scholarship website,Unknown,Unknown,Unknown,Unknown,Unknown,...,"Ulla, NC Accepted at a UNC System university, ...",Unknown,Unknown,The scholarship is renewable for up to four ye...,Unknown,Unknown,Applicants must also complete the FAFSA by Apr...,Unknown,2026-10-12,Automatically extracted candidate information....
1,DHS Friends Hilda Crocker Wilson Scholarship E...,Pending sponsor confirmation,Unknown,https://www.nccommunityfoundation.org/scholars...,Official scholarship website,Unknown,Unknown,Unknown,Unknown,Unknown,...,Unknown,Unknown,Unknown,Unknown,Unknown,Additional selection criteria include high sch...,Unknown,Unknown,2026-10-12,Automatically extracted candidate information....
2,Other NCCF Scholarships,Pending sponsor confirmation,Unknown,https://www.nccommunityfoundation.org/scholars...,Official scholarship website,Unknown,Unknown,Unknown,Unknown,Unknown,...,Preference is given to those living in or arou...,ESSAY PROMPT: Describe an experience that has ...,Raleigh Alumnae Delta Scholarship Endowment Ac...,Unknown,Lane Meyer Scholarship Fund Eligible students ...,Recipient(s) will be chosen on the basis of ac...,If you believe you are eligible for any these ...,Unknown,2026-10-12,Automatically extracted candidate information....
3,CASL Scholarship Fund in Memory of Lane Meyer,Pending sponsor confirmation,Unknown,https://www.nccommunityfoundation.org/scholars...,Official scholarship website,Unknown,Unknown,Unknown,Unknown,Unknown,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,"In addition to these qualifications, applicant...",Unknown,2026-10-12,Automatically extracted candidate information....
4,"Your Path to Recognition, Scholarship Opportun...",National Society of High School Scholars,Unknown,https://www.nshss.org/scholarships/,Official scholarship website,Unknown,Unknown,Unknown,Unknown,Unknown,...,Qualifying for college scholarships involves a...,Stand out in your personal essay by staying tr...,3) Prepare Required Documents: Such as officia...,3) Prepare Required Documents: Such as officia...,Look beyond academic achievements; consider sc...,High school and college scholarships are avail...,Please note that applicants may not win more t...,If you are awarded a scholarship while in your...,2026-10-12,Automatically extracted candidate information....


checking sponsor elligebility-my project requires A–M sponsors for North Carolina scholarships and N–Z sponsors for national scholarships.

In [81]:

def check_assignment_eligibility(row):

    provider = row["Scholarship provider/organization"]
    scope = row["National vs. state/local"]

    if (
        provider in ["Unknown", "Pending sponsor confirmation"]
        or scope == "Unknown"
    ):
        return "Needs verification"

    field = str(row["Field/area of interest"]).lower()

    if scope == "State/local":
        return check_sponsor_range(provider, "A", "M")

    if scope == "National":
        sponsor_ok = check_sponsor_range(provider, "N", "Z")

        field_ok = any(
            word in field
            for word in [
                "health", "nursing", "medical",
                "medicine", "pre-med"
            ]
        )

        if field == "unknown":
            return "Needs verification"

        return sponsor_ok and field_ok

    return "Needs verification"

def check_sponsor_range(provider, start, end):

    if pd.isna(provider) or provider == "Unknown":
        return False

    provider = str(provider).strip()

    if provider.lower().startswith("the "):
        provider = provider[4:]

    if not provider:
        return False

    first_letter = provider[0].upper()

    return start <= first_letter <= end


def check_assignment_eligibility(row):

    provider = row["Scholarship provider/organization"]

    scope = row["National vs. state/local"]

    field = str(
        row["Field/area of interest"]
    ).lower()

    if scope == "State/local":
        return check_sponsor_range(
            provider, "A", "M"
        )

    elif scope == "National":

        sponsor_ok = check_sponsor_range(
            provider, "N", "Z"
        )

        field_ok = any(
            keyword in field
            for keyword in [
                "health",
                "nursing",
                "medical",
                "medicine",
                "pre-med"
            ]
        )

        return sponsor_ok and field_ok

    return False


df["Assignment eligible"] = df.apply(
    check_assignment_eligibility,
    axis=1
)

display(df[[
    "Scholarship name",
    "Scholarship provider/organization",
    "Assignment eligible"
]])


,Scholarship name,Scholarship provider/organization,Assignment eligible
0,M&J Scholarship,M&J Scholarship donors,False
1,DHS Friends Hilda Crocker Wilson Scholarship E...,Pending sponsor confirmation,False
2,Other NCCF Scholarships,Pending sponsor confirmation,False
3,CASL Scholarship Fund in Memory of Lane Meyer,Pending sponsor confirmation,False
4,"Your Path to Recognition, Scholarship Opportun...",National Society of High School Scholars,False
5,NSHSS STEAM Scholarship,National Society of High School Scholars,False
6,Carteret County Retired School Personnel Schol...,Carteret County Retired School Personnel,False


In [82]:

def check_sponsor_range(provider, start, end):

    if pd.isna(provider) or provider == "Unknown":
        return False

    provider = str(provider).strip()

    if provider.lower().startswith("the "):
        provider = provider[4:]

    if not provider:
        return False

    first_letter = provider[0].upper()

    return start <= first_letter <= end


def check_assignment_eligibility(row):

    provider = row["Scholarship provider/organization"]

    scope = row["National vs. state/local"]

    field = str(
        row["Field/area of interest"]
    ).lower()

    if scope == "State/local":
        return check_sponsor_range(
            provider, "A", "M"
        )

    elif scope == "National":

        sponsor_ok = check_sponsor_range(
            provider, "N", "Z"
        )

        field_ok = any(
            keyword in field
            for keyword in [
                "health",
                "nursing",
                "medical",
                "medicine",
                "pre-med"
            ]
        )

        return sponsor_ok and field_ok

    return False


df["Assignment eligible"] = df.apply(
    check_assignment_eligibility,
    axis=1
)

display(df[[
    "Scholarship name",
    "Scholarship provider/organization",
    "Assignment eligible"
]])


,Scholarship name,Scholarship provider/organization,Assignment eligible
0,M&J Scholarship,M&J Scholarship donors,False
1,DHS Friends Hilda Crocker Wilson Scholarship E...,Pending sponsor confirmation,False
2,Other NCCF Scholarships,Pending sponsor confirmation,False
3,CASL Scholarship Fund in Memory of Lane Meyer,Pending sponsor confirmation,False
4,"Your Path to Recognition, Scholarship Opportun...",National Society of High School Scholars,False
5,NSHSS STEAM Scholarship,National Society of High School Scholars,False
6,Carteret County Retired School Personnel Schol...,Carteret County Retired School Personnel,False


Remove duplicate scholarships

In [83]:

def normalize_url(url):

    if pd.isna(url):
        return ""

    url, _ = urldefrag(str(url))

    parsed = urlparse(url)

    return (
        parsed.netloc.lower().removeprefix("www.")
        + parsed.path.rstrip("/").lower()
    )


df["Normalized URL"] = (
    df["Official scholarship URL"]
    .apply(normalize_url)
)

duplicates = df[
    df.duplicated(
        subset=["Normalized URL"],
        keep=False
    )
]

print("Potential duplicate records:")
display(duplicates)

df = df.drop_duplicates(
    subset=["Normalized URL"]
).copy()

df = df.drop(
    columns=["Normalized URL"]
)


Potential duplicate records:


,Scholarship name,Scholarship provider/organization,Provider type,Official scholarship URL,Discovery source,State/region,National vs. state/local,Eligible grade levels,Middle school / high school / both,Scholarship category,...,Recommendation required,Transcript/test score required,Community service requirement,Leadership requirement,Other major eligibility requirements,Application method,Verification date,Notes,Assignment eligible,Normalized URL


Step 15 — Export the dataset to Excel
This creates an Excel workbook with your 29 required fields and a separate sheet for source notes.

In [84]:

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

filename = "North_Carolina_Scholarship_Data.xlsx"

# Export only the required 29 fields
export_df = df[columns].copy()

with pd.ExcelWriter(
    filename,
    engine="openpyxl"
) as writer:

    export_df.to_excel(
        writer,
        sheet_name="Scholarship Data",
        index=False
    )

    notes_df = pd.DataFrame({
        "Item": [
            "Project",
            "State",
            "State sponsor range",
            "National sponsor range",
            "National scholarship theme",
            "Verification status"
        ],
        "Description": [
            "Scholarship Data Collection",
            "North Carolina",
            "A-M",
            "N-Z",
            "Health, nursing, and pre-med",
            "Automated candidate data; manual verification required"
        ]
    })

    notes_df.to_excel(
        writer,
        sheet_name="Sources & Notes",
        index=False
    )

    ws = writer.sheets["Scholarship Data"]

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    for cell in ws[1]:

        cell.fill = PatternFill(
            "solid",
            fgColor="17365D"
        )

        cell.font = Font(
            bold=True,
            color="FFFFFF"
        )

        cell.alignment = Alignment(
            wrap_text=True,
            vertical="center"
        )

    ws.row_dimensions[1].height = 35

    for col in range(1, len(columns) + 1):

        letter = get_column_letter(col)

        ws.column_dimensions[letter].width = 24

print("Excel workbook created:", filename)


Excel workbook created: North_Carolina_Scholarship_Data.xlsx


Downloading the Excel workbook from Google Colab

In [85]:

from google.colab import files

files.download(
    "North_Carolina_Scholarship_Data.xlsx"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>